Not necessary GRPO, but just the GRPO loss adopted into SoRL to replace the "select best abstraction" gadget

In [1]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# Disable MPS for stability
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper

from sorl.trainer_ablate import SoRLTrainerv2, SoRLTrainerv3
from sorl.trainer_ablate import SoRLConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Using device: cpu


In [2]:
# Initialize SoRL model
from sorl.sorl_wrapper import SorlModelWrapper
model_name = "Qwen/Qwen3-0.6B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name) 

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [3]:
# ============================================================
# SoRL v3 Training Kernel — all logic inline, easy to hack
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import time, os, json
from sorl.sorl_trainer import sorl_search, corrupt_abstract_tokens, ortho_loss
from sorl.sorl_wrapper import SorlModelWrapper
from data.pt_dataset import get_dataset, evaluate_accuracy, collate_fn

# ---- Config (edit freely) ----
cfg = dict(
    # Model
    model_name   = "Qwen/Qwen3-0.6B",
    abs_vocab    = 128,

    # Data
    dataset      = "gsm8k",
    max_length   = 256,
    batch_size   = 2,

    # SoRL search
    K            = 4,
    num_rollouts = 4,
    max_iters    = 2,
    temperature  = 1.0,
    mem_span_abs = 1792,
    mem_span_traj= 1792,

    # Corruption
    corrupt_method = "shuffle",
    corrupt_ratio  = 0.3,

    # Loss weights
    alpha_traj       = 1.0,     # p(s|a)
    alpha_contrastive= 1.0,     # hinge loss
    alpha_abs        = 0.5,     # p(a|s)
    gamma            = 0.5,     # hinge margin

    # Randomization (set to None to disable, or give a value/range)
    random_K         = None,    # e.g. (2, 4, 6, 8) — choices for K per batch
    strip_suffix     = None,    # e.g. (0.1, 1.0) — keep_frac range
    compress_prefix  = None,    # e.g. (0.0, 0.8) — compress_frac range
    random_mem_span  = None,    # e.g. (64, 1792) — memory_span_abs range

    # Optimizer
    lr           = 1e-5,
    emb_lr_mult  = 1.0,
    weight_decay = 0.01,
    max_grad_norm= 1.0,
    warmup_steps = 50,
    cooldown_frac= 0.4,

    # Training
    num_epochs   = 3,
    grad_accum   = 4,
    log_every    = 10,
    eval_every   = 99999,
    eval_samples = 100,
    save_every   = 99999,
    output_dir   = "./ckpt/v3_test",
)

print("Config:")
for k, v in cfg.items():
    print(f"  {k}: {v}")

Config:
  model_name: Qwen/Qwen3-0.6B
  abs_vocab: 128
  dataset: gsm8k
  max_length: 256
  batch_size: 2
  K: 4
  num_rollouts: 4
  max_iters: 2
  temperature: 1.0
  mem_span_abs: 1792
  mem_span_traj: 1792
  corrupt_method: shuffle
  corrupt_ratio: 0.3
  alpha_traj: 1.0
  alpha_contrastive: 1.0
  alpha_abs: 0.5
  gamma: 0.5
  random_K: None
  strip_suffix: None
  compress_prefix: None
  random_mem_span: None
  lr: 1e-05
  emb_lr_mult: 1.0
  weight_decay: 0.01
  max_grad_norm: 1.0
  warmup_steps: 50
  cooldown_frac: 0.4
  num_epochs: 3
  grad_accum: 4
  log_every: 10
  eval_every: 99999
  eval_samples: 100
  save_every: 99999
  output_dir: ./ckpt/v3_test


In [4]:
# ============================================================
# GRPO-style SoRL: train on ALL rollouts weighted by advantage
# ============================================================
#
# Current SoRL v3 pipeline:
#   1. Insert abstract tokens → N rollouts via model.recursion()
#   2. select_best_sequences() → train on best rollout only
#   3. Loss = α_traj * p(s|a) + α_contrastive * hinge + α_abs * p(a|s) + ...
#
# GRPO alternative (with PPO-style clipping):
#   1. Same rollout procedure, but also capture old log-probs π_old(a_t)
#   2. Compute per-rollout advantage: A_i = -(loss_i - group_mean) / group_std
#   3. At training time, compute new log-probs π_θ(a_t) under current policy
#   4. ratio = π_θ / π_old = exp(log_π_θ - log_π_old)
#   5. Clipped surrogate: L = -min(ratio * A, clip(ratio, 1-ε, 1+ε) * A)
#   6. No zipf/ortho needed if GRPO naturally prevents vocab collapse
#
# Validation criterion:
#   If this notebook run avoids "vocabulary collapse" (top-1 token < 90%)
#   WITHOUT zipf & ortho regularization, that validates GRPO > best-of-N.

import torch
import torch.nn as nn
import torch.nn.functional as F
from sorl.sorl_trainer import (
    infer_insert_mask, expand_prompt_len, insert_tokens_with_padding,
    sorl_search, corrupt_abstract_tokens, ortho_loss,
)
from data.pt_dataset import get_dataset, evaluate_accuracy, collate_fn


# ---- Helper: compute per-token log-probs at trajectory / abstract positions ----

def _compute_token_logprobs(logits, labels, mask, vocab_slice):
    """
    Compute per-token log-probs for positions indicated by `mask`.
    vocab_slice: (lo, hi) — logits[:, :, lo:hi] are the valid range.
    Returns: per-token log-probs (B, L), zeros at masked-out positions.
    """
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()

    # Slice logits to valid vocab range
    sliced = shift_logits.clone()
    sliced[..., :vocab_slice[0]] = -float("inf")
    if vocab_slice[1] is not None:
        sliced[..., vocab_slice[1]:] = -float("inf")

    log_probs = F.log_softmax(sliced, dim=-1)  # (B, L-1, V)

    # Safe labels for gather (replace invalid positions with a valid index)
    safe_labels = shift_labels.clone()
    safe_labels[~mask.bool()] = max(vocab_slice[0], 0)  # any valid index in range
    per_tok_lp = log_probs.gather(2, safe_labels.unsqueeze(-1)).squeeze(-1)  # (B, L-1)
    per_tok_lp = per_tok_lp * mask  # zero out non-target positions
    return per_tok_lp


def _build_masks(data, attn_mask, prompt_len, base_vocab):
    """Build traj_mask and abs_mask for shifted positions."""
    shift_attn = attn_mask[..., 1:].contiguous().clone()
    if prompt_len is not None:
        seq_idx = torch.arange(shift_attn.size(1), device=shift_attn.device).unsqueeze(0)
        shift_attn[seq_idx < (prompt_len.unsqueeze(1) - 1)] = 0
    levels = (data >= base_vocab).long()[:, 1:]
    traj_mask = (levels == 0).float() * shift_attn.float()
    abs_mask = (1 - (levels == 0).float()) * shift_attn.float()
    return traj_mask, abs_mask


# ---- Rollouts: now also returns old log-probs for ratio computation ----

def sorl_rollouts(model, input_ids, attention_mask, prompt_len, pad_token_id,
                  n=4, K=4, max_iterations=2,
                  memory_span_abs=1792, memory_span_traj=1792,
                  temperature=1.0, response_only_abs=False):
    """
    Run SoRL rollouts WITHOUT selection.
    Returns all N rollouts per sample + old log-probs for GRPO ratio.
    """
    base_vocab = int(model.vocab_sizes[0].item())

    insert_mask = infer_insert_mask(
        input_ids, K, attention_mask,
        prompt_len=prompt_len if response_only_abs else None,
    )
    expanded_prompt_len = expand_prompt_len(prompt_len, insert_mask)
    expanded_data, expanded_mask = insert_tokens_with_padding(
        input_ids, attention_mask, insert_mask, model.vocab_sizes[0], pad_token_id,
    )

    repeated_data = expanded_data.repeat_interleave(n, dim=0)     # (B*n, L')
    repeated_mask = expanded_mask.repeat_interleave(n, dim=0)
    repeated_pl   = expanded_prompt_len.repeat_interleave(n, dim=0)

    with torch.no_grad():
        search_data, search_ppt, _ = model.recursion(
            repeated_data, repeated_mask,
            max_iterations=max_iterations,
            memory_span_abs=memory_span_abs,
            memory_span_traj=memory_span_traj,
            temperature=temperature,
            prompt_len=repeated_pl,
        )

        # Capture old log-probs π_old for abstract AND trajectory tokens
        # We need a fresh forward pass on the final search_data to get clean logits
        old_out = model(
            input_ids=search_data, attention_mask=repeated_mask,
            memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj,
        )
        old_logits = old_out.logits

        traj_mask, abs_mask = _build_masks(search_data, repeated_mask, repeated_pl, base_vocab)

        # Old log-probs at trajectory positions (base vocab only)
        old_traj_lp = _compute_token_logprobs(
            old_logits, search_data, traj_mask, vocab_slice=(0, base_vocab))
        # Old log-probs at abstract positions (abstract vocab only)
        old_abs_lp = _compute_token_logprobs(
            old_logits, search_data, abs_mask, vocab_slice=(base_vocab + 1, None))

    # Per-rollout scalar loss (for advantage computation)
    valid_mask = (search_ppt != 0).float()
    per_rollout_loss = search_ppt.sum(dim=1) / valid_mask.sum(dim=1).clamp(min=1)

    return (search_data, search_ppt, per_rollout_loss,
            expanded_mask, expanded_prompt_len, n,
            old_traj_lp, old_abs_lp)


# ---- Advantage computation (unchanged) ----

def compute_grpo_advantages(per_rollout_loss, batch_size, n):
    """
    GRPO advantage: negative of (loss - group_mean) / group_std.
    Lower loss = better rollout = positive advantage.
    """
    losses = per_rollout_loss.view(batch_size, n)
    group_mean = losses.mean(dim=1, keepdim=True)
    group_std  = losses.std(dim=1, keepdim=True).clamp(min=1e-6)
    advantages = -(losses - group_mean) / group_std
    return advantages.view(-1)


# ---- GRPO loss with PPO-style clipping ----

def grpo_clipped_loss(model, all_data, all_mask, all_prompt_len,
                      advantages, old_traj_lp, old_abs_lp,
                      base_vocab, alpha_traj=1.0, alpha_abs=0.5,
                      clip_eps=0.2, memory_span_abs=1792, memory_span_traj=1792):
    """
    GRPO with PPO-style clipped surrogate objective.

    For each rollout i with advantage A_i:
      ratio_t = exp(log π_θ(a_t) - log π_old(a_t))   per token
      surr1 = ratio_t * A_i
      surr2 = clip(ratio_t, 1-ε, 1+ε) * A_i
      loss_t = -min(surr1, surr2)                      pessimistic bound

    We compute this separately for trajectory and abstract positions,
    then combine with alpha weights.
    """
    B_n, L = all_data.shape

    # Forward pass (with gradient)
    outputs = model(
        input_ids=all_data, attention_mask=all_mask,
        memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj,
    )
    logits = outputs.logits

    traj_mask, abs_mask = _build_masks(all_data, all_mask, all_prompt_len, base_vocab)

    # New log-probs under current policy
    new_traj_lp = _compute_token_logprobs(
        logits, all_data, traj_mask, vocab_slice=(0, base_vocab))
    new_abs_lp = _compute_token_logprobs(
        logits, all_data, abs_mask, vocab_slice=(base_vocab + 1, None))

    # Expand advantages to per-token: (B*n,) → (B*n, 1)
    adv = advantages.unsqueeze(1)  # (B*n, 1)

    # ---- Clipped surrogate for trajectory tokens ----
    traj_ratio = (new_traj_lp - old_traj_lp).exp()  # π_θ / π_old per token
    traj_surr1 = traj_ratio * adv
    traj_surr2 = traj_ratio.clamp(1 - clip_eps, 1 + clip_eps) * adv
    # Pessimistic bound: take the min (since we want to maximize, loss = -min)
    traj_clipped = -torch.min(traj_surr1, traj_surr2) * traj_mask
    traj_loss = traj_clipped.sum() / traj_mask.sum().clamp(min=1)

    # ---- Clipped surrogate for abstract tokens ----
    abs_ratio = (new_abs_lp - old_abs_lp).exp()
    abs_surr1 = abs_ratio * adv
    abs_surr2 = abs_ratio.clamp(1 - clip_eps, 1 + clip_eps) * adv
    abs_clipped = -torch.min(abs_surr1, abs_surr2) * abs_mask
    abs_loss = abs_clipped.sum() / abs_mask.sum().clamp(min=1)

    # ---- Combined ----
    loss = alpha_traj * traj_loss + alpha_abs * abs_loss

    # Diagnostics (detached)
    with torch.no_grad():
        avg_traj_ratio = (traj_ratio * traj_mask).sum() / traj_mask.sum().clamp(min=1)
        avg_abs_ratio = (abs_ratio * abs_mask).sum() / abs_mask.sum().clamp(min=1)
        clip_frac_traj = ((traj_ratio - 1).abs() > clip_eps).float()
        clip_frac_traj = (clip_frac_traj * traj_mask).sum() / traj_mask.sum().clamp(min=1)
        clip_frac_abs = ((abs_ratio - 1).abs() > clip_eps).float()
        clip_frac_abs = (clip_frac_abs * abs_mask).sum() / abs_mask.sum().clamp(min=1)

    diagnostics = {
        "traj_loss": traj_loss.detach(),
        "abs_loss": abs_loss.detach(),
        "avg_traj_ratio": avg_traj_ratio,
        "avg_abs_ratio": avg_abs_ratio,
        "clip_frac_traj": clip_frac_traj,
        "clip_frac_abs": clip_frac_abs,
    }

    return loss, diagnostics


print("GRPO kernel defined (with PPO-style clipping).")

GRPO kernel defined (with PPO-style clipping).


In [ ]:
# ============================================================
# GRPO Training Loop (with inner-loop updates for meaningful clipping)
# ============================================================
#
# Key insight: PPO/GRPO clipping only works when π_θ drifts from π_old.
# We collect rollouts once (π_old), then do K_INNER gradient steps on the
# same batch. Each step makes π_θ diverge from π_old, and clipping prevents
# the policy from changing too much.
#
from torch.utils.data import DataLoader

# ---- Load dataset ----
train_ds = get_dataset(cfg["dataset"], split="train", tokenizer=tokenizer, max_length=cfg["max_length"])
val_ds   = get_dataset(cfg["dataset"], split="test",  tokenizer=tokenizer, max_length=cfg["max_length"])
dl = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, collate_fn=collate_fn)
pad_token_id = tokenizer.pad_token_id
base_vocab = int(model.vocab_sizes[0].item())

# ---- Optimizer (separate embed/lm_head LR) ----
emb_params, other_params = [], []
for name, p in model.named_parameters():
    if "embed_tokens" in name or "lm_head" in name:
        emb_params.append(p)
    else:
        other_params.append(p)

optimizer = torch.optim.AdamW([
    {"params": other_params, "lr": cfg["lr"]},
    {"params": emb_params,   "lr": cfg["lr"] * cfg["emb_lr_mult"]},
], weight_decay=cfg["weight_decay"])

# ---- LR schedule (linear warmup + cosine decay) ----
total_steps = len(dl) * cfg["num_epochs"]  # each batch = 1 outer step (K_INNER inner steps)

def get_lr(step):
    if step < cfg["warmup_steps"]:
        return cfg["lr"] * step / max(cfg["warmup_steps"], 1)
    frac = (step - cfg["warmup_steps"]) / max(total_steps - cfg["warmup_steps"], 1)
    return cfg["lr"] * 0.5 * (1 + np.cos(np.pi * frac))

# ---- Vocab collapse tracker ----
def measure_vocab_usage(model, dl_iter, base_vocab, K=4, n_batches=5):
    """Count abstract token usage across a few batches to detect collapse."""
    model.eval()
    abs_counts = torch.zeros(model.vocab_sizes[-1], device=device)
    with torch.no_grad():
        for i, batch in enumerate(dl_iter):
            if i >= n_batches:
                break
            ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            pl = batch["prompt_len"].to(device)
            insert_mask = infer_insert_mask(ids, K, attn)
            exp_pl = expand_prompt_len(pl, insert_mask)
            exp_data, exp_mask = insert_tokens_with_padding(ids, attn, insert_mask, model.vocab_sizes[0], pad_token_id)
            search_data, _, _ = model.recursion(
                exp_data, exp_mask, max_iterations=2,
                memory_span_abs=cfg["mem_span_abs"], memory_span_traj=cfg["mem_span_traj"],
                temperature=cfg["temperature"], prompt_len=exp_pl,
            )
            abs_tokens = search_data[search_data >= base_vocab] - base_vocab
            abs_counts.scatter_add_(0, abs_tokens.long(), torch.ones_like(abs_tokens, dtype=torch.float))
    model.train()
    total = abs_counts.sum().item()
    if total == 0:
        return 0, 1.0, 0
    sorted_counts = abs_counts.sort(descending=True).values
    top1_pct = (sorted_counts[0] / total * 100).item()
    top3_pct = (sorted_counts[:3].sum() / total * 100).item()
    n_used = (abs_counts > 0).sum().item()
    return n_used, top1_pct, top3_pct

# ---- GRPO config ----
CLIP_EPS = 0.2    # PPO clip range
K_INNER  = 4      # inner-loop gradient steps per rollout batch

# ---- Training ----
history = {"step": [], "loss": [], "traj": [], "abs": [],
           "clip_frac_traj": [], "clip_frac_abs": [],
           "avg_traj_ratio": [], "avg_abs_ratio": [],
           "vocab_used": [], "top1_pct": [], "top3_pct": []}

model.train()
global_step = 0

print(f"Total outer steps: {total_steps} | batches/epoch: {len(dl)}")
print(f"N rollouts: {cfg['num_rollouts']} | K: {cfg['K']} | clip_eps: {CLIP_EPS} | K_inner: {K_INNER}")
print("=" * 70)

t0 = time.time()
for epoch in range(cfg["num_epochs"]):
    for batch_idx, batch in enumerate(dl):
        input_ids      = batch["input_ids"].to(device)
        attention_mask  = batch["attention_mask"].to(device)
        prompt_len     = batch["prompt_len"].to(device)

        # LR schedule
        lr = get_lr(global_step)
        optimizer.param_groups[0]["lr"] = lr
        optimizer.param_groups[1]["lr"] = lr * cfg["emb_lr_mult"]

        # ---- 1. Rollouts with π_old (no grad, frozen snapshot) ----
        # old_traj_lp / old_abs_lp are log-probs under the CURRENT model
        # before any inner-loop updates. They stay fixed for all K_INNER steps.
        (all_data, all_ppt, per_rollout_loss,
         exp_mask, exp_pl, n,
         old_traj_lp, old_abs_lp) = sorl_rollouts(
            model, input_ids, attention_mask, prompt_len, pad_token_id,
            n=cfg["num_rollouts"], K=cfg["K"],
            max_iterations=cfg["max_iters"],
            memory_span_abs=cfg["mem_span_abs"],
            memory_span_traj=cfg["mem_span_traj"],
            temperature=cfg["temperature"],
        )

        # 2. Compute advantages (fixed for all inner steps)
        B = input_ids.shape[0]
        advantages = compute_grpo_advantages(per_rollout_loss, B, n)

        # 3. Expand masks for all rollouts
        exp_mask_rep = exp_mask.repeat_interleave(n, dim=0)
        exp_pl_rep   = exp_pl.repeat_interleave(n, dim=0)

        # ---- 4. Inner loop: K_INNER gradient steps on same batch ----
        # After each step, π_θ drifts from π_old → ratio diverges from 1.0
        # → clipping becomes meaningful
        for k_inner in range(K_INNER):
            optimizer.zero_grad(set_to_none=True)

            loss, diag = grpo_clipped_loss(
                model, all_data, exp_mask_rep, exp_pl_rep,
                advantages, old_traj_lp, old_abs_lp,
                base_vocab,
                alpha_traj=cfg["alpha_traj"], alpha_abs=cfg["alpha_abs"],
                clip_eps=CLIP_EPS,
                memory_span_abs=cfg["mem_span_abs"],
                memory_span_traj=cfg["mem_span_traj"],
            )

            loss.backward()

            if cfg["max_grad_norm"] > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
            optimizer.step()

        global_step += 1

        # ---- Logging (after all inner steps) ----
        if (batch_idx + 1) % cfg["log_every"] == 0:
            elapsed = time.time() - t0
            frac = max(global_step, 1) / max(total_steps, 1)
            eta = elapsed / frac * (1 - frac) if frac > 0 else 0
            eta_m, eta_s = divmod(int(eta), 60)

            print(f"ep {epoch + (batch_idx+1)/len(dl):.2f}/{cfg['num_epochs']} "
                  f"| step {global_step}/{total_steps} "
                  f"| loss={loss.item():.4f} "
                  f"traj={diag['traj_loss'].item():.4f} abs={diag['abs_loss'].item():.4f} "
                  f"| ratio t={diag['avg_traj_ratio'].item():.3f} a={diag['avg_abs_ratio'].item():.3f} "
                  f"| clip t={diag['clip_frac_traj'].item():.2f} a={diag['clip_frac_abs'].item():.2f} "
                  f"| lr={lr:.2e} | eta={eta_m}m{eta_s:02d}s")

            history["step"].append(global_step)
            history["loss"].append(loss.item())
            history["traj"].append(diag["traj_loss"].item())
            history["abs"].append(diag["abs_loss"].item())
            history["clip_frac_traj"].append(diag["clip_frac_traj"].item())
            history["clip_frac_abs"].append(diag["clip_frac_abs"].item())
            history["avg_traj_ratio"].append(diag["avg_traj_ratio"].item())
            history["avg_abs_ratio"].append(diag["avg_abs_ratio"].item())

        # ---- Vocab collapse check ----
        if global_step > 0 and global_step % cfg["eval_every"] == 0:
            check_dl = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, collate_fn=collate_fn)
            n_used, top1, top3 = measure_vocab_usage(model, iter(check_dl), base_vocab, K=cfg["K"])
            print(f"  >> VOCAB: {n_used}/{cfg['abs_vocab']} used | top1={top1:.1f}% top3={top3:.1f}%")
            history["vocab_used"].append(n_used)
            history["top1_pct"].append(top1)
            history["top3_pct"].append(top3)

        del loss, all_data, all_ppt, per_rollout_loss, advantages, old_traj_lp, old_abs_lp
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"=== Epoch {epoch+1} complete ===")

    # End-of-epoch vocab check
    check_dl = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, collate_fn=collate_fn)
    n_used, top1, top3 = measure_vocab_usage(model, iter(check_dl), base_vocab, K=cfg["K"])
    print(f"  >> VOCAB: {n_used}/{cfg['abs_vocab']} used | top1={top1:.1f}% top3={top3:.1f}%")
    history["vocab_used"].append(n_used)
    history["top1_pct"].append(top1)
    history["top3_pct"].append(top3)

print("\nTraining complete!")
print(f"Final vocab: {n_used}/{cfg['abs_vocab']} | top1={top1:.1f}% | top3={top3:.1f}%")

Total outer steps: 11211 | batches/epoch: 3737
N rollouts: 4 | K: 4 | clip_eps: 0.2 | K_inner: 4


In [ ]:
# ============================================================
# Visualization: Loss curves + Vocab collapse tracking
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Loss curve
ax = axes[0, 0]
ax.plot(history["step"], history["loss"], label="total loss", alpha=0.8)
ax.plot(history["step"], history["traj"], label="traj loss", alpha=0.8)
ax.plot(history["step"], history["abs"], label="abs loss", alpha=0.8)
ax.set_xlabel("Step"); ax.set_ylabel("Loss")
ax.set_title("GRPO Training Loss")
ax.legend(); ax.grid(True, alpha=0.3)

# Vocab usage over time
ax = axes[0, 1]
if history["vocab_used"]:
    ax.plot(history["vocab_used"], marker='o', label="# tokens used")
    ax.axhline(y=cfg["abs_vocab"], color='r', linestyle='--', alpha=0.5, label=f'total={cfg["abs_vocab"]}')
    ax.set_xlabel("Checkpoint"); ax.set_ylabel("Unique abstract tokens")
    ax.set_title("Vocab Usage (collapse = low)")
    ax.legend(); ax.grid(True, alpha=0.3)

# Top-1 / Top-3 concentration
ax = axes[1, 0]
if history["top1_pct"]:
    ax.plot(history["top1_pct"], marker='o', label="top-1 %")
    ax.plot(history["top3_pct"], marker='s', label="top-3 %")
    ax.axhline(y=90, color='r', linestyle='--', alpha=0.5, label='collapse threshold (90%)')
    ax.set_xlabel("Checkpoint"); ax.set_ylabel("% of total usage")
    ax.set_title("Token Concentration (collapse = high)")
    ax.legend(); ax.grid(True, alpha=0.3)

# Summary text
ax = axes[1, 1]
ax.axis('off')
if history["vocab_used"]:
    final_used = history["vocab_used"][-1]
    final_top1 = history["top1_pct"][-1]
    final_top3 = history["top3_pct"][-1]
    collapsed = final_top1 > 90
    verdict = "COLLAPSED" if collapsed else "DIVERSE"
    color = "red" if collapsed else "green"
    summary = (
        f"GRPO SoRL — Final Results\n"
        f"{'='*30}\n\n"
        f"Vocab used:  {final_used} / {cfg['abs_vocab']}\n"
        f"Top-1 token: {final_top1:.1f}%\n"
        f"Top-3 tokens: {final_top3:.1f}%\n\n"
        f"Verdict: {verdict}\n\n"
        f"No zipf loss, no ortho loss.\n"
        f"If diverse → GRPO > best-of-N"
    )
    ax.text(0.1, 0.5, summary, transform=ax.transAxes, fontsize=12,
            verticalalignment='center', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.15))

plt.tight_layout()
plt.savefig("./figure/grpo_vs_bestofn.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved to ./figure/grpo_vs_bestofn.png")

In [ ]:
# ============================================================
# Evaluate: NL accuracy vs K=4 accuracy (same as ablation evals)
# ============================================================

model.eval()

# NL accuracy (K=None, no abstract tokens)
print("Evaluating NL accuracy (no abstract tokens)...")
nl_result = evaluate_accuracy(
    model, tokenizer, val_ds, device,
    num_samples=cfg["eval_samples"],
    eval_K=None,
)
print(f"  NL accuracy: {nl_result['accuracy']*100:.1f}%")

# K=4 accuracy (with abstract tokens)
print(f"Evaluating K={cfg['K']} accuracy (with abstract tokens)...")
k4_result = evaluate_accuracy(
    model, tokenizer, val_ds, device,
    num_samples=cfg["eval_samples"],
    eval_K=cfg["K"],
)
print(f"  K={cfg['K']} accuracy: {k4_result['accuracy']*100:.1f}%")

gap = (k4_result['accuracy'] - nl_result['accuracy']) * 100
print(f"\n  Gap (K=4 - NL): {gap:+.1f}pp")
print(f"  {'Abstractions HELP' if gap >= 0 else 'Abstractions HURT'}")

model.train()
print("\nDone.")

NameError: name 'dl' is not defined

In [ ]:
# Idea 2. 
# SoRL -> Distillation
# Pseudo code 1.  
# alpha_zipf = 1.0, alpha_ortho = 1.0, temperature = 1.0, num_rollouts= 4
# sorl_trainer.train()
# sorl_trainer.config.temperature = 0.0
# sorl_trainer.config.alpha_zipf = 0.0
# sorl_trainer.config.alpha_ortho = 0.0
# sorl_trainer.config.num_rollouts = 1
# sorl_trainer.train()  -> this is the most naive way, we assume abstraction choice is stable, once we set temperature to zero 
#                          we also assume abstraction accuracy will improve when we remove the search process from it

# Above is the easy to implement v1. a hard 'distillation' v2 requires a copied model 
# ref_model = sorl_trainer.model
# use 'ref_model' to perform the abstraction generation, and setting the configs similarly

# Validation criterion: 
# We'd need proper accuracy validation, it's also interesting to see if vocabulary collapse after the v1 style distillation


In [ ]:
# Idea 3. 
# Differentiable Search Process
# This is very interesting, as I am curious how we can achieve it.
#  
# v1. When we "generate" abstraction via decoding (with some temperature), we'd like to replace 
# index by a STE value that contains gradient, sth like "index + logit - stop_gradient(logit)"? 
# The hope here, is that this approach (applied to the last 'iteration' of Jacobi decoding / recursion) 
# can makes the abstraction search "differentiable"
# 
# Validation criterion: no need to incorporate into SoRL yet, just use a toy module (one layer MLP that predict logit)
# then we show a temperatured decoding sample can be differentiable suffices in notebook

